# Post-Tuning Evaluation

This notebook evaluates the fine-tuned MiniCPM phonetic evaluator against the baseline ASR results. The baseline ASR marks only exact transcript matches as correct; the tuned SLM is used as a second-stage judge for strict-match failures, where child speech patterns, lisps, plurals, or near-phonetic transcripts may still be acceptable.

In [ ]:
from pathlib import Path

import modal
import pandas as pd

BASELINE_RESULTS = Path("data/baseline_results.csv")
if not BASELINE_RESULTS.exists():
    BASELINE_RESULTS = Path("../data/baseline_results.csv")
MODAL_APP_NAME = "read-along-ai-inference"
MODAL_FUNCTION_NAME = "run_minicpm_evaluator"

df = pd.read_csv(BASELINE_RESULTS)
df["Strict Match"] = df["Strict Match"].astype(bool)
df.head()

In [ ]:
failures = df.loc[~df["Strict Match"]].copy()
print(f"Strict-match failures to review with MiniCPM: {len(failures)}")
failures[["File", "Target Word", "ASR Transcript", "Strict Match"]]

## MiniCPM As A Phonetic Judge

The tuned model receives only the target word and ASR transcript, then outputs `True` or `False`. A `True` verdict means the transcript is an acceptable phonetic match for the intended word, even if the baseline ASR transcript was not an exact string match.

In [ ]:
def modal_function(app_name: str, function_name: str):
    lookup = getattr(modal.Function, "lookup", None)
    if lookup is not None:
        return lookup(app_name, function_name)
    return modal.Function.from_name(app_name, function_name)


evaluator = modal_function(MODAL_APP_NAME, MODAL_FUNCTION_NAME)
slm_verdicts = []

for _, row in failures.iterrows():
    verdict = evaluator.remote(row["Target Word"], row["ASR Transcript"])
    slm_verdicts.append(str(verdict).strip())

failures["SLM Verdict"] = slm_verdicts
failures["SLM Accepts"] = failures["SLM Verdict"].str.casefold().eq("true")
failures[["Target Word", "ASR Transcript", "SLM Verdict", "SLM Accepts"]]

In [ ]:
baseline_correct = int(df["Strict Match"].sum())
slm_recovered = int(failures["SLM Accepts"].sum())
new_correct = baseline_correct + slm_recovered
total = len(df)

summary = pd.DataFrame(
    [
        {
            "System": "Baseline ASR exact match",
            "Correct": baseline_correct,
            "Total": total,
            "Accuracy": baseline_correct / total,
        },
        {
            "System": "ASR + tuned MiniCPM phonetic judge",
            "Correct": new_correct,
            "Total": total,
            "Accuracy": new_correct / total,
        },
    ]
)

comparison = failures.assign(
    **{
        "Baseline Verdict": "False",
        "New System Verdict": failures["SLM Verdict"],
    }
)[["Target Word", "ASR Transcript", "Baseline Verdict", "New System Verdict"]]

display(
    summary.style.format({"Accuracy": "{:.1%}"}).hide(axis="index")
)
display(
    comparison.style.set_caption("Strict-match failures reviewed by tuned MiniCPM")
)